# 示例: EnKF参数率定与数据同化工作流

**相关脚本:**
- `examples/calibrate_with_enkf.py`
- `examples/plot_enkf_results.py`

## 目的
此示例用于演示项目的第三部分功能：使用集合卡尔曼滤波（EnKF）进行参数率定和数据同化。它将两个独立的脚本组合成一个完整的工作流，展示了：
1.  如何使用“增广状态向量”技术，将模型参数与状态变量一同放入EnKF框架中进行优化。
2.  如何实时地将观测流量数据同化进模型，动态地校正模型状态和参数。
3.  同化过程如何显著改善流量模拟效果（相比于无同化的“开环”模拟）。
4.  模型参数如何随着时间的推移，从一个不准确的初始猜测值，逐渐收敛到更优、更稳定的值。

## 第1部分: 运行EnKF同化

**脚本:** `examples/calibrate_with_enkf.py`

这部分是工作流的核心。我们按顺序执行以下步骤：
1.  **定义模型推进函数**: 创建一个函数 `model_forward_augmented`，它接受一个包含模型状态（如土壤含水量S）和参数（如S_max, k_q）的“增广状态向量”，并返回更新后的状态向量和模拟的流量。
2.  **初始化EnKF**: 设置集合成员数量 `N_ENSEMBLE` 和观测误差 `R`。
3.  **创建初始集合**: 基于对参数的初步猜测和不确定性，生成一个状态-参数集合。集合中的每个成员都是对真实状态和参数的一种可能“猜测”。
4.  **运行同化循环**: 在每个时间步：
    a. **预测 (Forecast)**: 使用模型推进函数，将集合中的每个成员向前推进一个时间步。
    b. **分析 (Analysis)**: 使用该时间步的观测流量，根据卡尔曼滤波的公式，对预测后的集合进行校正，得到一个更接近“真实”的新集合。
5.  **运行开环模拟**: 作为对比，我们还运行一个没有数据同化的普通模拟（称为“开环”模拟），以展示同化的效果。
6.  **保存结果**: 将观测流量、开环模拟流量、同化后流量，以及参数随时间演变的数据保存到CSV文件中，供后续分析。

In [ ]:
import numpy as np
import pandas as pd
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SimpleRunoffModule
from hydro_model.routing import SimpleRouting
from hydro_model.enkf import EnsembleKalmanFilter

def model_forward_augmented(state_vector, rainfall, pet, area_km2):
    S, Q_s, S_max, k_q, k_s, c_loss = state_vector
    params = {
        'S_max': max(1, S_max),
        'k_q': max(0.01, min(1.0, k_q)),
        'k_s': max(0.01, min(1.0, k_s)),
        'c_loss': max(0, c_loss)
    }
    runoff_module = SimpleRunoffModule(**params)
    runoff_module.S = S
    routing_module = SimpleRouting(**params)
    routing_module.Q_s = Q_s
    model = HydrologicalModel(runoff_module, routing_module)
    runoff_mm = model.run(rainfall, pet)
    simulated_flow_m3s = (runoff_mm / 1000) * (area_km2 * 1e6) / (24 * 3600)
    param_noise = np.random.normal(0, [0.1, 0.001, 0.001, 0.0001])
    new_params = np.array([params['S_max'], params['k_q'], params['k_s'], params['c_loss']]) + param_noise
    new_state_vector = np.array([runoff_module.S, routing_module.Q_s, *new_params])
    return new_state_vector, simulated_flow_m3s

def run_open_loop(model_params, rainfall_data, pet_data, area_km2):
    runoff_module = SimpleRunoffModule(**model_params)
    routing_module = SimpleRouting(**model_params)
    model = HydrologicalModel(runoff_module, routing_module)
    results = []
    for rain, pet in zip(rainfall_data, pet_data):
        runoff_mm = model.run(rain, pet)
        flow_m3s = (runoff_mm / 1000) * (area_km2 * 1e6) / (24 * 3600)
        results.append(flow_m3s)
    return np.array(results)

# --- 1. 设置和加载数据 ---
rainfall_df = pd.read_csv('../data/rainfall.csv', index_col='date', parse_dates=True)
pet_df = pd.read_csv('../data/pet.csv', index_col='date', parse_dates=True)
observed_flow_df = pd.read_csv('../data/observed_flow.csv', index_col='date', parse_dates=True)

catchment_area = 150 + 200 + 120 # km^2
rainfall = rainfall_df['rainfall_1'].values
pet = pet_df['pet'].values
observed_flow = observed_flow_df['flow_m3s'].values

# --- 2. 初始化EnKF ---
N_ENSEMBLE = 50
R = 10**2
enkf = EnsembleKalmanFilter(n_ensemble=N_ENSEMBLE)

# --- 3. 创建初始集合 ---
initial_guess = {'S': 10.0, 'Q_s': 5.0, 'S_max': 180.0, 'k_q': 0.7, 'k_s': 0.15, 'c_loss': 0.06}
initial_uncertainty = {'S': 5.0, 'Q_s': 2.0, 'S_max': 50.0, 'k_q': 0.2, 'k_s': 0.1, 'c_loss': 0.02}
ensemble = np.random.normal(loc=list(initial_guess.values()), scale=list(initial_uncertainty.values()), size=(N_ENSEMBLE, len(initial_guess))).T
enkf.initialize(initial_states=ensemble)

# --- 4. 运行同化循环 ---
T = len(rainfall)
assimilated_flows = np.zeros(T)
state_history = np.zeros((T, len(initial_guess)))

print("运行EnKF同化...")
for t in range(T):
    forecast_obs = enkf.forecast(model_forward=model_forward_augmented, rainfall=rainfall[t], pet=pet[t], area_km2=catchment_area)
    enkf.analysis(observation=observed_flow[t], forecast_observations=forecast_obs, R=R)
    mean_state = enkf.states.mean(axis=1)
    state_history[t, :] = mean_state
    _, mean_flow = model_forward_augmented(mean_state, rainfall[t], pet[t], catchment_area)
    assimilated_flows[t] = mean_flow
print("同化完成。")

# --- 5. 运行开环模拟 ---
print("运行开环模拟用于对比...")
open_loop_params = {k: v for k, v in initial_guess.items() if k not in ['S', 'Q_s']}
open_loop_flows = run_open_loop(open_loop_params, rainfall, pet, catchment_area)

# --- 6. 保存结果 ---
results_df = pd.DataFrame({'observed_flow': observed_flow, 'open_loop_flow': open_loop_flows, 'assimilated_flow': assimilated_flows}, index=observed_flow_df.index)
param_names = ['S', 'Q_s', 'S_max', 'k_q', 'k_s', 'c_loss']
params_df = pd.DataFrame(state_history, columns=param_names, index=observed_flow_df.index)

output_dir = '../examples/results/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_flow_path = os.path.join(output_dir, 'enkf_flow_results.csv')
output_params_path = os.path.join(output_dir, 'enkf_parameter_evolution.csv')
results_df.to_csv(output_flow_path)
params_df.to_csv(output_params_path)

print(f"结果已保存到 '{output_flow_path}' 和 '{output_params_path}'")

## 第2部分: 可视化结果

**脚本:** `examples/plot_enkf_results.py`

现在我们有了结果，接下来将它们可视化，以直观地评估同化的效果。

1.  **流量对比图**: 我们将观测流量、开环模拟流量和同化后的流量绘制在同一张图上。这可以非常清楚地展示数据同化对改善模拟精度的巨大作用。
2.  **参数演变图**: 我们将增广状态向量中的几个关键模型参数（`S_max`, `k_q`, `k_s`, `c_loss`）随时间的变化过程绘制出来。这可以帮助我们判断参数是否收敛到了一个稳定的、更优的值。

In [ ]:
import matplotlib.pyplot as plt

def plot_flow_comparison():
    df = pd.read_csv(output_flow_path, index_col='date', parse_dates=True)
    plt.figure(figsize=(15, 7))
    plt.plot(df.index, df['observed_flow'], 'k.', label='Observed Flow')
    plt.plot(df.index, df['open_loop_flow'], 'r--', label='Open-Loop (No Assimilation)')
    plt.plot(df.index, df['assimilated_flow'], 'b-', label='EnKF Assimilated Flow')
    plt.title('Comparison of Hydrological Model Flows')
    plt.xlabel('Date')
    plt.ylabel('Flow (m³/s)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plot_path = os.path.join(output_dir, 'enkf_flow_comparison.png')
    plt.savefig(plot_path)
    print(f"流量对比图已保存到 {plot_path}")
    plt.show()

def plot_parameter_evolution():
    df = pd.read_csv(output_params_path, index_col='date', parse_dates=True)
    params_to_plot = ['S_max', 'k_q', 'k_s', 'c_loss']
    fig, axes = plt.subplots(nrows=len(params_to_plot), ncols=1, figsize=(12, 10), sharex=True)
    fig.suptitle('Evolution of Model Parameters via EnKF', fontsize=16)
    for i, param in enumerate(params_to_plot):
        axes[i].plot(df.index, df[param], label=f'Estimated {param}')
        axes[i].set_ylabel(param)
        axes[i].legend(loc='upper right')
        axes[i].grid(True, linestyle='--', alpha=0.6)
    axes[-1].set_xlabel('Date')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plot_path = os.path.join(output_dir, 'enkf_parameter_convergence.png')
    plt.savefig(plot_path)
    print(f"参数演变图已保存到 {plot_path}")
    plt.show()

plot_flow_comparison()
plot_parameter_evolution()